In [61]:
# ==========================================
# Load Silver Fact Orders
# ==========================================

fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

print("Fact Orders:", fact_orders.count())

In [62]:
# ==========================================
# Create Date Dimension
# ==========================================

from pyspark.sql.functions import *

dim_date = (
    fact_orders
    .select("order_date")
    .distinct()
    .withColumn("day", dayofmonth("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("month_name", date_format("order_date", "MMMM"))
    .withColumn("quarter", quarter("order_date"))
    .withColumn("year", year("order_date"))
    .withColumn("weekday", date_format("order_date", "EEEE"))
    .withColumn(
        "is_weekend",
        dayofweek("order_date").isin([1, 7])
    )
)

print("Distinct Dates:", dim_date.count())

In [63]:
display(
    dim_date.orderBy("order_date")
)

In [64]:
dim_date.write \
    .mode("overwrite") \
    .parquet(
        "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_date"
    )

In [65]:
dim_date_check = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_date"
)

print("Rows:", dim_date_check.count())

display(
    dim_date_check.orderBy("order_date")
)